In [11]:
import pandas as pd
from pathlib import Path
import os
import sys
import numpy as np
import librosa
import librosa.display
#import msaf
import eyed3
import time
from IPython.display import Audio
music_dir = Path('../../music/halloween2k24')
songs = list(music_dir.glob('*.mp3'))

# Testing MSAF
MSAF is the Music Structure Analysis Framework. 

In [98]:
song = '../../music/halloween2k24/Mindchatter - Young Folks.mp3'
print(msaf.features_registry)

{'cqt': <class 'msaf.features.CQT'>, 'mfcc': <class 'msaf.features.MFCC'>, 'pcp': <class 'msaf.features.PCP'>, 'tonnetz': <class 'msaf.features.Tonnetz'>, 'tempogram': <class 'msaf.features.Tempogram'>}


In [81]:
def process_song(song, feature='mfcc'):
    songlength = eyed3.load(song).info.time_secs
    print(f'Song length: {songlength} seconds')
    t0 = time.time()
    boundaries, labels = msaf.process(song, feature=feature)
    t1 = time.time()
    print(f'Processing took {t1-t0} seconds')
    return boundaries, labels

In [111]:
boundaries, labels = process_song(song, feature='tempogram')

Invalid POPM play count: less than 32 bits.


Song length: 277.82 seconds
Processing took 20.03004002571106 seconds


In [112]:
print(f'Estimated boundaries: {boundaries}')
print(f'Estimated labels: {labels}')

Estimated boundaries: [  0.           2.73995465  34.27265306  64.27283447  91.62594104
 118.97904762 148.97922902 184.27356009 217.01369615 249.56807256
 277.7106576  277.75401361]
Estimated labels: [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 0.0]


In [113]:
def chunkify(boundaries, labels, song):
    waveform, sr = librosa.load(song)
    print(f'Sampling rate: {sr}')
    chunks = []
    for i in range(len(boundaries)-1):
        start = int(boundaries[i]*sr)
        end = int(boundaries[i+1]*sr)
        chunk = waveform[start:end]
        chunks.append(chunk)
    return chunks

In [114]:
chunks = chunkify(boundaries, labels, song)
_, sr = librosa.load(song)
print('sr2', sr)

Sampling rate: 22050
sr2 22050


MSAF is not great. Best bet will be to use pre-computed segments based on a deeper pipeline (i.e., an LSTM) and then using the Spotifizer to keep track of where we're at in the song. Next I'm going to try all-in-one.[https://github.com/mir-aidj/all-in-one]

In [151]:
# !pip install git+https://github.com/CPJKU/madmom  # install the latest madmom directly from GitHub
# !pip install allin1  # install this package

In [2]:
import allin1

In [3]:
song = '../../music/halloween2k24/Mindchatter - Young Folks.mp3'

In [5]:
result = allin1.analyze(song, demix_dir='../scripts/demix')

=> Found 0 tracks already analyzed and 1 tracks to analyze.
=> Found 1 tracks already demixed, 0 to demix.
=> Found 0 spectrograms already extracted, 1 to extract.


Analyzing Mindchatter - Young Folks.mp3:   0%|          | 0/1 [00:00<?, ?it/s]| 2024-09-07 10:55:17,268 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dqkrpb`, which is deprecated in favor of `natten.functional.na1d_qk`. Please consider switching, as this op will be removed soon.
| 2024-09-07 10:55:17,581 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dav`, which is deprecated in favor of `natten.functional.na1d_av`. Please consider switching, as this op will be removed soon.
| 2024-09-07 10:55:17,651 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dqkrpb`, which is deprecated in favor of `natten.functional.na1d_qk`. Please consider switching, as this op will be removed soon.
| 2024-09-07 10:55:17,960 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dav`, which is deprecated in favor of `natten.functional.na1d_av`. Plea

In [6]:
result

AnalysisResult(path=PosixPath('/Users/f004swn/Documents/creative/music/halloween2k24/Mindchatter - Young Folks.mp3'), bpm=136, beats=[7.31, 7.75, 8.19, 8.64, 9.07, 9.52, 9.96, 10.4, 10.84, 11.28, 11.72, 12.16, 12.6, 13.05, 13.49, 13.93, 14.37, 14.81, 15.25, 15.69, 16.13, 16.58, 17.02, 17.46, 17.9, 18.34, 18.78, 19.22, 19.66, 20.1, 20.54, 20.99, 21.43, 21.87, 22.31, 22.75, 23.19, 23.64, 24.08, 24.52, 24.96, 25.4, 25.84, 26.28, 26.72, 27.17, 27.61, 28.05, 28.49, 28.93, 29.37, 29.81, 30.25, 30.7, 31.14, 31.58, 32.02, 32.46, 32.9, 33.34, 33.77, 34.22, 34.67, 35.11, 35.54, 35.99, 36.43, 36.87, 37.31, 37.75, 38.19, 38.63, 39.07, 39.52, 39.96, 40.4, 40.84, 41.28, 41.72, 42.16, 42.6, 43.05, 43.49, 43.93, 44.37, 44.81, 45.25, 45.69, 46.13, 46.58, 47.02, 47.46, 47.9, 48.34, 48.78, 49.22, 49.66, 50.11, 50.55, 50.99, 51.43, 51.87, 52.31, 52.75, 53.19, 53.64, 54.07, 54.52, 54.96, 55.4, 55.84, 56.28, 56.72, 57.17, 57.6, 58.05, 58.49, 58.93, 59.37, 59.81, 60.25, 60.69, 61.13, 61.58, 62.02, 62.46, 62.

In [12]:
songs

[PosixPath('../../music/halloween2k24/Cherub - Doses & Mimosas.mp3'),
 PosixPath('../../music/halloween2k24/Glass Animals - Hazey.mp3'),
 PosixPath("../../music/halloween2k24/it's murph, YDG, Sorana - Down Low - YDG Remix.mp3"),
 PosixPath('../../music/halloween2k24/Albin Myers - Take Me Down (2020 VIP) - Radio Edit.mp3'),
 PosixPath('../../music/halloween2k24/Taylor Swift, Post Malone - Fortnight (feat. Post Malone).mp3'),
 PosixPath('../../music/halloween2k24/Mr Little Jeans - Good Mistake.mp3'),
 PosixPath('../../music/halloween2k24/TroyBoi - Bellz.mp3'),
 PosixPath('../../music/halloween2k24/Andruss - Tranquilao.mp3'),
 PosixPath('../../music/halloween2k24/Ida Corr, Fedde Le Grand - Let Me Think About It.mp3'),
 PosixPath('../../music/halloween2k24/Crystal Castles - Vanished.mp3'),
 PosixPath('../../music/halloween2k24/Mindchatter - Young Folks.mp3'),
 PosixPath("../../music/halloween2k24/The Sponges - Space Funk '75.mp3"),
 PosixPath('../../music/halloween2k24/Dr. Fresch, Marten H

In [14]:
for s, song in enumerate([str(i) for i in songs]):
    result2 = allin1.analyze(song, demix_dir='../scripts/demix', include_activations=True, include_embeddings=True)

=> Found 0 tracks already analyzed and 1 tracks to analyze.
=> Found 1 tracks already demixed, 0 to demix.
=> Found 1 spectrograms already extracted, 0 to extract.


Analyzing Cherub - Doses & Mimosas.mp3:   0%|          | 0/1 [00:00<?, ?it/s]| 2024-09-07 11:05:12,419 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dqkrpb`, which is deprecated in favor of `natten.functional.na1d_qk`. Please consider switching, as this op will be removed soon.
| 2024-09-07 11:05:13,913 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dav`, which is deprecated in favor of `natten.functional.na1d_av`. Please consider switching, as this op will be removed soon.
| 2024-09-07 11:05:14,206 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dqkrpb`, which is deprecated in favor of `natten.functional.na1d_qk`. Please consider switching, as this op will be removed soon.
| 2024-09-07 11:05:16,030 | [[ natten.functional ]] [ WARNING ]: You're calling NATTEN op `natten.functional.natten1dav`, which is deprecated in favor of `natten.functional.na1d_av`. Pleas

KeyboardInterrupt: 

Ok this is working crazy good! https://arxiv.org/pdf/2307.16425

# Mapping 

In [16]:
import json

In [17]:
# save result
with open('result.json', 'w') as f:
    json.dump(result, f)

TypeError: Object of type AnalysisResult is not JSON serializable

In [18]:
from allin1.helpers import save_results

In [19]:
save_results(result, out_dir='results')